# Part 1. Equation of a Slime

How many late days are you using for this assignment?

4

In [136]:
# Imports section
import pandas as pd


## 1. Loading the dataset

In [137]:
# Using pandas load the dataset
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)

ds = pd.read_csv('science_data_large.csv')
ds.head(15)

,Temperature °C,Mols KCL,Size nm^3
0,469,647,6.244743e+05
1,403,694,5.779610e+05
2,302,975,6.196847e+05
3,779,916,1.460449e+06
4,901,18,4.325726e+04
5,545,637,7.124634e+05
6,660,519,7.006960e+05
7,143,869,2.718260e+05
8,89,461,8.919803e+04
9,294,776,4.770210e+05


In [138]:
ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 23.6 KB


## 2. Splitting the dataset

In [139]:
# Take the pandas dataset and split it into our features (X) and label (y)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 

from sklearn.model_selection import train_test_split

X = ds[['Temperature °C', 'Mols KCL']]
y = ds['Size nm^3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## 3. Perform a Linear Regression

In [140]:
# Use sklearn to train a model on the training set

# Create a sample datapoint and predict the output of that sample with the trained model

# Report on the score for that model, in your own words (markdown, not code) explain what the score means

# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX

from sklearn.linear_model import LinearRegression

lin_reg_model = LinearRegression().fit(X_train, y_train)

sample_datapoint = pd.DataFrame([{'Temperature °C': 600, 'Mols KCL': 800}])
prediction = lin_reg_model.predict(sample_datapoint).round(5)
print("prediction: ", prediction)

score = round(lin_reg_model.score(X_test, y_test), 5)
print("score: ", score)

coeff = lin_reg_model.coef_.round(5)
intercept = lin_reg_model.intercept_.round(5)
print("coefficients: ", coeff)
print("intercept: ", intercept)

prediction:  [936452.42163]
score:  0.85525
coefficients:  [ 866.14641 1032.69507]
intercept:  -409391.47958


Write the linear equation of a slime: (example equation: $E = mc^2$)

$h(x) = -409391.47958 + 866.14641x_1 + 1032.69507x_2$

Report on score and explain meaning: The score for the model is 0.85525 so that means that around 85.5% of the variance for our label, the size of the slime, is explained by our features, the temperature and moles of KCL. Overall, this score is suggesting that our model does a good job at predicting the size of the slime given a temperature and moles of KCL.

## 4. Use Cross Validation

In [142]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42

# Report on their finding and their significance

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lin_reg_model, X, y, cv=kf).round(5)
print("cross validation scores: ", cv_scores)

cross validation scores:  [0.86152 0.82742 0.87195 0.88166 0.85609]


Write findings here: The scores for each shuffle is fairly consistent as it ranges from 0.82742 to 0.88166. This suggests our model isn't over fitting as it's doing well using different subsets of our training data. 


## 5. Using Polynomial Regression

In [144]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
# Perform k-fold cross validation (as above)

# Report on the metrics and output the resultant equation as you did in Part 3.
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

poly_reg_model = Pipeline([('poly', PolynomialFeatures(degree=2)),
                  ('linear', LinearRegression())])

poly_reg_model.fit(X_train, y_train)

sample_datapoint = pd.DataFrame([{'Temperature °C': 600, 'Mols KCL': 800}])
prediction = poly_reg_model.predict(sample_datapoint).round(5)
print("prediction: ", prediction)

score = round(poly_reg_model.score(X_test, y_test), 5)
print("score: ", score)

coeff = poly_reg_model.named_steps['linear'].coef_.round(5)
intercept = poly_reg_model.named_steps['linear'].intercept_.round(5)
print("coefficients: ", coeff)
print("intercept: ", intercept)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(poly_reg_model, X, y, cv=kf).round(5)
print("cross validation scores: ", cv_scores)


prediction:  [985485.71429]
score:  1
coefficients:  [ 0.      12.      -0.       0.       2.       0.02857]
intercept:  2e-05
cross validation scores:  [1. 1. 1. 1. 1.]


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

$h(x) = 0.00002 + 12x_1+ 2x_1x_2+ 0.02857x_{2}^{2}$

Report on the score and interpret: The model has a score of 1 on both the test data and cross validation on the training data. This means our model is making perfect predictions on the size of the slime given the temperature and mol of KCL. Since our dataset has 1000 entries, which isn't considered small, over fitting probably isn't an issue so the relationship between the features and labels might genuinely be represented by our equation above.

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [145]:
# Load the dataset. Then train and evaluate the classification models.
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import numpy as np

ckd_ds = pd.read_csv('ckd_feature_subset.csv')
X = ckd_ds[['age', 'bp', 'wbcc', 'appet_poor', 'appet_good', 'rbcc']]
y = ckd_ds['Target_ckd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [164]:
#logistic regression model
log_reg_model = LogisticRegression()
log_reg_cv_scores = cross_val_score(log_reg_model, X, y, cv=5)

log_reg_avg = np.mean(log_reg_cv_scores)
log_reg_std = np.std(log_reg_cv_scores)
print("logistic regression cross validation scores: ", log_reg_cv_scores.round(5))
print(f"logistic regression: avg = {log_reg_avg:.5f}, std = {log_reg_std:.5f}")

#support vector machines model
svc_model = SVC()
svc_cv_scores = cross_val_score(svc_model, X, y, cv=5)

svc_avg = np.mean(svc_cv_scores)
svc_std = np.std(svc_cv_scores)
print("svc cross validation scores: ", svc_cv_scores.round(5))
print(f"svc: avg = {svc_avg:.5f}, std = {svc_std:.5f}")

#k-nearest neighbors model
knn_model = KNeighborsClassifier()
knn_cv_scores = cross_val_score(knn_model, X, y, cv=5)

knn_avg = np.mean(knn_cv_scores)
knn_std = np.std(knn_cv_scores)
print("knn cross validation scores: ", knn_cv_scores.round(5))
print(f"knn: avg = {knn_avg:.5f}, std = {knn_std:.5f}")

#neural networks model (multi-layer perceptron)
mlp_model = MLPClassifier(max_iter = 1000)
mlp_cv_scores = cross_val_score(mlp_model, X, y, cv=5)

mlp_avg = np.mean(mlp_cv_scores)
mlp_std = np.std(mlp_cv_scores)
print("mlp cross validation scores: ", mlp_cv_scores.round(5))
print(f"mlp: avg = {mlp_avg:.5f}, std = {mlp_std:.5f}")

logistic regression cross validation scores:  [0.87097 0.83871 0.87097 0.93333 0.76667]
logistic regression: avg = 0.85613, std = 0.05424
svc cross validation scores:  [0.90323 1.      0.87097 1.      0.86667]
svc: avg = 0.92817, std = 0.05999
knn cross validation scores:  [0.90323 0.96774 0.90323 0.96667 0.9    ]
knn: avg = 0.92817, std = 0.03189
mlp cross validation scores:  [0.96774 0.93548 0.90323 0.96667 0.93333]
mlp: avg = 0.94129, std = 0.02404


## Results and Conclusion for Classification Experiments
| Model              | Average (Avg) | Standard Deviation (Std) |
|--------------------|---------------|--------------------------|
| Logistic Regression | 0.85613       | 0.05424                  |
| SVC                | 0.92817       | 0.05999                  |
| k-Nearest Neighbors | 0.92817       | 0.03189                  |
| MLP                | 0.94129       | 0.02404                  |

It seems like MLP performed best with the highest average score and lowest standard deviation. KNN came in second since although it's average is tied to SVC, its standard deviation is the second lowest. The reason why I believe MLP performed the best is because by default, if I don't specify the number of hidden layers, the model I created has a single hidden layer with 100 neurons, which makes it capable of capturing complex relationships and patterns that simpler models, such as logistic regression, can miss.

In [180]:
#config 1
num_hidden_layers = 1
neurons_per_layer = 100
solver = 'adam' #by default

mlp_model_1 = MLPClassifier(hidden_layer_sizes=(neurons_per_layer), max_iter=1000, random_state=42)
mlp_cv_scores_1 = cross_val_score(mlp_model_1, X, y, cv=5, scoring='accuracy')
mlp_avg_1 = np.mean(mlp_cv_scores_1)
mlp_std_1 = np.std(mlp_cv_scores_1)
print("mlp 1 cross validation scores: ", mlp_cv_scores_1.round(5))
print(f"mlp 1: avg = {mlp_avg_1:.5f}, std = {mlp_std_1:.5f}")
print(f"mlp 1 configurations: hidden layers = {num_hidden_layers}, neurons_per_layer = {neurons_per_layer}, "
f"solver = {solver}.\n")

#config 2
num_hidden_layers = 4
neurons_per_layer = 50
mlp_model_2 = MLPClassifier(hidden_layer_sizes=(50, 50, 50, 50), max_iter=1000, random_state=42)
mlp_cv_scores_2 = cross_val_score(mlp_model_2, X, y, cv=5, scoring='accuracy')
mlp_avg_2 = np.mean(mlp_cv_scores_2)
mlp_std_2 = np.std(mlp_cv_scores_2)
print("mlp 2 cross validation scores: ", mlp_cv_scores_2.round(5))
print(f"mlp 2: avg = {mlp_avg_2:.5f}, std = {mlp_std_2:.5f}")
print(f"mlp 2 configurations: hidden layers = {num_hidden_layers}, neurons_per_layer = {neurons_per_layer}, "
f"solver = {solver}.\n")

#config 3
num_hidden_layers = 1
neurons_per_layer = 100
solver = 'sgd'
mlp_model_3 = MLPClassifier(hidden_layer_sizes=(neurons_per_layer), 
                            solver='sgd', max_iter=10000, random_state=42)
mlp_cv_scores_3 = cross_val_score(mlp_model_3, X, y, cv=5, scoring='accuracy')
mlp_avg_3 = np.mean(mlp_cv_scores_3)
mlp_std_3 = np.std(mlp_cv_scores_3)
print("mlp 3 cross validation scores: ", mlp_cv_scores_3.round(5))
print(f"mlp 3: avg = {mlp_avg_3:.5f}, std = {mlp_std_3:.5f}")
print(f"mlp 3 configurations: hidden layers = {num_hidden_layers}, neurons_per_layer = {neurons_per_layer}, "
f"solver = {solver}.")

mlp 1 cross validation scores:  [0.96774 0.93548 0.90323 0.96667 0.93333]
mlp 1: avg = 0.94129, std = 0.02404
mlp 1 configurations: hidden layers = 1, neurons_per_layer = 100, solver = adam.

mlp 2 cross validation scores:  [1.      0.96774 0.93548 1.      0.96667]
mlp 2: avg = 0.97398, std = 0.02420
mlp 2 configurations: hidden layers = 4, neurons_per_layer = 50, solver = adam.

mlp 3 cross validation scores:  [0.83871 0.83871 0.87097 0.93333 0.76667]
mlp 3: avg = 0.84968, std = 0.05401
mlp 3 configurations: hidden layers = 1, neurons_per_layer = 100, solver = sgd.


## Results and Conclusion for Neural Network Experiments
| Configuration | Hidden Layers | Neurons per Layer | Solver | Average Accuracy (Avg) | Standard Deviation (Std) |
|---------------|---------------|-------------------|--------|------------------------|--------------------------|
| MLP 1         | 1             | 100               | adam   | 0.94129                | 0.02404                 |
| MLP 2         | 4             | 50                | adam   | 0.97398                | 0.02420                 |
| MLP 3         | 1             | 100               | sgd    | 0.84968                | 0.05401                 |

We can observe from MLP 1 and MLP 2 that when hidden layers increase, even when the neurons per layer decreases, the average accuracy increases. However, this isn't the parameter in our configuration that caused the greatest difference. It's actually the solver parameter that tells the model which optimization algorithm to use for weight optimization that created the biggest difference. We can see from MLP 1 and MLP 3 by changing the solver from adam to sgd (stochastic gradient descent), the average accuracy decreases by approximately 10% and the standard deviation roughly doubles.
